In [2]:
# VERIFICATION CELL — 24 Aug 2026. Four read-only checks before Notebook 05.
# Nothing is saved or changed; this cell only prints evidence.

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PROC = Path("..").resolve() / "data/processed"

# Load audited data and rebuild the 70/15/15 split exactly as notebook 04 does.
df = pd.read_parquet(DATA_PROC / "01_audit.parquet")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)
df["split"] = np.where(df["Timestamp"] <= t70, "train",
              np.where(df["Timestamp"] <= t85, "val", "test"))
assert df["split"].value_counts().to_dict() == {
    "train": 3554957, "val": 761749, "test": 761639}, "Split mismatch"
print("Split reproduced OK\n")

# ---------- CHECK A: wind-down rule (Option A) verified by execution ----------
# Median daily volume across complete calendar days inside the training window (1-6 Sep);
# any day below 5% of that median is a wind-down day, excluded from evaluation.
df["day"] = df["Timestamp"].dt.date
daily = df.groupby("day").size()
train_days = [d for d in daily.index if pd.Timestamp(d) >= pd.Timestamp("2022-09-01")
              and pd.Timestamp(d) <= pd.Timestamp("2022-09-06")]
M = daily.loc[train_days].median()
threshold = 0.05 * M
winddown_days = set(daily[daily < threshold].index)
print(f"A) Median training-day volume M = {M:,.1f}  (expect 482,369.5)")
print(f"   Threshold 0.05*M = {threshold:,.1f}")
print(f"   Wind-down days: {sorted(winddown_days)}")
test = df[df["split"] == "test"]
test_main = test[~test["day"].isin(winddown_days)]
wd = test[test["day"].isin(winddown_days)]
print(f"   Test-main: n={len(test_main):,}  illicit={test_main['Is Laundering'].sum():,}"
      f"  (expect 760,531 / 906)")
print(f"   Wind-down: n={len(wd):,}  illicit={wd['Is Laundering'].sum():,}"
      f"  (expect 1,108 / 655)\n")

# ---------- CHECK B: are laundering rows almost always same-currency? ----------
for label, name in [(1, "Illicit"), (0, "Legitimate")]:
    sub = df[df["Is Laundering"] == label]
    same_ccy = (sub["Payment Currency"] == sub["Receiving Currency"]).mean()
    same_amt = (sub["Amount Paid"] == sub["Amount Received"]).mean()
    print(f"B) {name}: same currency {same_ccy:.2%} | identical amounts {same_amt:.2%}")
print()

# ---------- CHECK C: does any account ID appear under more than one bank? ----------
# Notebook 04 uses the account string alone as the vertex name. If one ID maps to
# several banks, the graph silently merged distinct accounts.
send = df[["From Bank", "Account"]].rename(columns={"From Bank": "bank", "Account": "acct"})
recv = df[["To Bank", "Account.1"]].rename(columns={"To Bank": "bank", "Account.1": "acct"})
pairs = pd.concat([send, recv], ignore_index=True)
pairs["acct"] = pairs["acct"].astype(str)
pairs = pairs.drop_duplicates()
banks_per_acct = pairs.groupby("acct")["bank"].nunique()
n_collisions = int((banks_per_acct > 1).sum())
print(f"C) Unique account IDs (whole dataset): {banks_per_acct.shape[0]:,}")
print(f"   IDs appearing under >1 bank: {n_collisions:,}"
      f"  ({'GRAPH IS SAFE' if n_collisions == 0 else 'PROBLEM - accounts merged'})\n")

# ---------- CHECK D: unseen accounts + staleness strata on test-main ----------
# Accounts with no training-window activity get imputed profiles (U3 Option A).
# Count them, then size the three strata against the degeneracy rule
# (any stratum < 1,000 transactions or < 10 illicit -> counts only).
tr = df[df["split"] == "train"]
train_accts = set(tr["Account"].astype(str)) | set(tr["Account.1"].astype(str))
print(f"D) Accounts in training window: {len(train_accts):,}  (expect 513,284)")

tm = test_main.copy()
seen_s = tm["Account"].astype(str).isin(train_accts)
seen_r = tm["Account.1"].astype(str).isin(train_accts)
tm["stratum"] = np.select(
    [seen_s & seen_r, seen_s ^ seen_r], ["both-active", "one-imputed"], "both-imputed")
strata = tm.groupby("stratum").agg(
    n=("Is Laundering", "size"), illicit=("Is Laundering", "sum"))
strata["degenerate"] = (strata["n"] < 1000) | (strata["illicit"] < 10)
print(strata, "\n")

all_accts = set(df["Account"].astype(str)) | set(df["Account.1"].astype(str))
unseen = len(all_accts) - len(train_accts)
print(f"   Total accounts anywhere: {len(all_accts):,}  ->  unseen in training: "
      f"{unseen:,} ({unseen/len(all_accts):.2%})")

Split reproduced OK

A) Median training-day volume M = 482,369.5  (expect 482,369.5)
   Threshold 0.05*M = 24,118.5
   Wind-down days: [datetime.date(2022, 9, 11), datetime.date(2022, 9, 12), datetime.date(2022, 9, 13), datetime.date(2022, 9, 14), datetime.date(2022, 9, 15), datetime.date(2022, 9, 16), datetime.date(2022, 9, 17), datetime.date(2022, 9, 18)]
   Test-main: n=760,531  illicit=906  (expect 760,531 / 906)
   Wind-down: n=1,108  illicit=655  (expect 1,108 / 655)

B) Illicit: same currency 100.00% | identical amounts 100.00%
B) Legitimate: same currency 98.58% | identical amounts 98.58%

C) Unique account IDs (whole dataset): 515,080
   IDs appearing under >1 bank: 8  (PROBLEM - accounts merged)

D) Accounts in training window: 513,284  (expect 513,284)
                   n  illicit  degenerate
stratum                                  
both-active   758982      878       False
both-imputed      58        0        True
one-imputed     1491       28       False 

   Total accou

In [3]:
# Cross-check for Check B: build a table of Payment Currency (rows)
# vs Receiving Currency (columns) for illicit transactions only.
# If every illicit transaction is same-currency, all counts sit on
# the diagonal and every off-diagonal cell is zero.

illicit = df[df["Is Laundering"] == 1]

ccy_table = pd.crosstab(illicit["Payment Currency"], illicit["Receiving Currency"])
print(ccy_table)
print()

# Count how many illicit rows fall OFF the diagonal (should be 0)
off_diagonal = (illicit["Payment Currency"] != illicit["Receiving Currency"]).sum()
print(f"Illicit rows where the two currencies differ: {off_diagonal}")

# And for contrast: the same count for legitimate rows (should be ~72,000)
legit = df[df["Is Laundering"] == 0]
off_diagonal_legit = (legit["Payment Currency"] != legit["Receiving Currency"]).sum()
print(f"Legitimate rows where the two currencies differ: {off_diagonal_legit:,}")

Receiving Currency  Australian Dollar  Bitcoin  Brazil Real  Canadian Dollar  \
Payment Currency                                                               
Australian Dollar                 127        0            0                0   
Bitcoin                             0       56            0                0   
Brazil Real                         0        0           57                0   
Canadian Dollar                     0        0            0              128   
Euro                                0        0            0                0   
Mexican Peso                        0        0            0                0   
Ruble                               0        0            0                0   
Rupee                               0        0            0                0   
Saudi Riyal                         0        0            0                0   
Shekel                              0        0            0                0   
Swiss Franc                         0   

In [4]:
# Save the currency-consistency evidence to outputs/tables/,
# so the finding is a recorded artefact rather than a screenshot.

OUT_TABLES = Path("..").resolve() / "outputs" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

# Full illicit currency table (all counts on the diagonal)
ccy_table.to_csv(OUT_TABLES / "ccy_table.csv")

# Headline numbers
summary = pd.DataFrame({
    "check": ["illicit same-currency", "illicit identical amounts",
              "legitimate same-currency", "illicit cross-currency rows",
              "legitimate cross-currency rows"],
    "value": ["100.00%", "100.00%", "98.58%", 0, 72170],
})
summary.to_csv(OUT_TABLES / "ccy_summary.csv", index=False)

print("Saved ccy_table.csv and ccy_summary.csv to outputs/tables/")

Saved ccy_table.csv and ccy_summary.csv to outputs/tables/


In [5]:
# Look at the account IDs that appear under more than one bank.
# Self-contained: rebuilds the bank/account pairs so it doesn't
# depend on any earlier cell having been run.

senders = df[["From Bank", "Account"]].rename(columns={"From Bank": "bank", "Account": "acct"})
receivers = df[["To Bank", "Account.1"]].rename(columns={"To Bank": "bank", "Account.1": "acct"})

pairs = pd.concat([senders, receivers], ignore_index=True)
pairs["acct"] = pairs["acct"].astype(str)
pairs = pairs.drop_duplicates()

banks_per_account = pairs.groupby("acct")["bank"].nunique()
colliding_ids = banks_per_account[banks_per_account > 1].index.tolist()
print("Colliding IDs:", colliding_ids)
print()

detail = pairs[pairs["acct"].isin(colliding_ids)].sort_values(["acct", "bank"])
print(detail.to_string(index=False))
print()

# How many transactions touch these IDs, and are any illicit?
touched = df[df["Account"].astype(str).isin(colliding_ids) |
             df["Account.1"].astype(str).isin(colliding_ids)]
print(f"Transactions touching a colliding ID: {len(touched):,}")
print(f"Of which illicit: {touched['Is Laundering'].sum():,}")

Colliding IDs: ['80A7FD400', '80A7FDE00', '80FA55EF0', '80FA56340', '81211BA20', '81211BC00', '8135B8200', '8135B8250']

  bank      acct
 27755 80A7FD400
 28248 80A7FD400
 27755 80A7FDE00
 28248 80A7FDE00
 13858 80FA55EF0
138832 80FA55EF0
 13858 80FA56340
138832 80FA56340
  1490 81211BA20
142574 81211BA20
  1490 81211BC00
142574 81211BC00
 27444 8135B8200
221731 8135B8200
 27444 8135B8250
221731 8135B8250

Transactions touching a colliding ID: 56
Of which illicit: 0


In [8]:
# Diagnose the median mismatch: is the file complete, and what are the
# actual daily counts? No asserts yet - we're looking, not testing.
import pandas as pd

df = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\01_audit.parquet",
                     columns=["Timestamp"])

print("Rows loaded:", len(df))                      # expect 5,078,345
print("Span:", df["Timestamp"].min(), "to", df["Timestamp"].max())

daily = df.groupby(df["Timestamp"].dt.date).size().sort_index()
print("\nDaily counts, all days:")
print(daily.to_string())

M = daily.median()
print(f"\nMedian = {M}, threshold (5%) = {0.05*M}")
print("Days below threshold:", [str(d) for d in daily[daily < 0.05*M].index])

Rows loaded: 5078345
Span: 2022-09-01 00:00:00 to 2022-09-18 16:18:00

Daily counts, all days:
Timestamp
2022-09-01    1114921
2022-09-02     754449
2022-09-03     207382
2022-09-04     207430
2022-09-05     482650
2022-09-06     482089
2022-09-07     482751
2022-09-08     482773
2022-09-09     654467
2022-09-10     208325
2022-09-11        396
2022-09-12        281
2022-09-13        184
2022-09-14        121
2022-09-15         46
2022-09-16         46
2022-09-17         23
2022-09-18         11

Median = 207406.0, threshold (5%) = 10370.300000000001
Days below threshold: ['2022-09-11', '2022-09-12', '2022-09-13', '2022-09-14', '2022-09-15', '2022-09-16', '2022-09-17', '2022-09-18']


In [10]:
# Diagnose the split-label alignment: was positional assignment valid?
import pandas as pd

df = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\01_audit.parquet",
                     columns=["Timestamp"])
split_idx = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\03_split_index.parquet")

print("Lengths:", len(df), len(split_idx))
print("\nsplit_idx columns:", list(split_idx.columns))
print("split_idx index:", split_idx.index[:5].tolist(), "...", split_idx.index[-5:].tolist())
print("df index:      ", df.index[:5].tolist(), "...", df.index[-5:].tolist())

# What the POSITIONAL pasting produced (the buggy view):
df_pos = df.copy()
df_pos["split"] = split_idx["split"].values
print("\nPositional assignment -> per-split date ranges:")
print(df_pos.groupby("split")["Timestamp"].agg(["min", "max", "size"]))

# What an INDEX-based join produces (the safe view):
df_join = df.join(split_idx["split"])
print("\nIndex-based join -> per-split date ranges:")
print(df_join.groupby("split")["Timestamp"].agg(["min", "max", "size"]))

Lengths: 5078345 5078345

split_idx columns: ['Timestamp', 'split']
split_idx index: [0, 1, 2, 3, 4] ... [5078340, 5078341, 5078342, 5078343, 5078344]
df index:       [0, 1, 2, 3, 4] ... [5078340, 5078341, 5078342, 5078343, 5078344]

Positional assignment -> per-split date ranges:
                      min                 max     size
split                                                 
test  2022-09-09 03:00:00 2022-09-18 16:18:00   761639
train 2022-09-01 00:00:00 2022-09-14 18:32:00  3554957
val   2022-09-07 14:30:00 2022-09-14 08:33:00   761749

Index-based join -> per-split date ranges:
                      min                 max     size
split                                                 
test  2022-09-09 03:00:00 2022-09-18 16:18:00   761639
train 2022-09-01 00:00:00 2022-09-14 18:32:00  3554957
val   2022-09-07 14:30:00 2022-09-14 08:33:00   761749


In [11]:
# Is 03_split_index self-consistent, and is the order mismatch the whole story?
import pandas as pd

df = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\01_audit.parquet",
                     columns=["Timestamp"])
split_idx = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\03_split_index.parquet")

print("split_idx's OWN per-split boundaries:")
print(split_idx.groupby("split")["Timestamp"].agg(["min", "max", "size"]))

print("\n01_audit sorted by time? ", df["Timestamp"].is_monotonic_increasing)
print("split_idx sorted by time?", split_idx["Timestamp"].is_monotonic_increasing)
print("Timestamps identical row-by-row?",
      (split_idx["Timestamp"].values == df["Timestamp"].values).all())
print("Same multiset of timestamps?",
      split_idx["Timestamp"].sort_values().reset_index(drop=True)
      .equals(df["Timestamp"].sort_values().reset_index(drop=True)))

split_idx's OWN per-split boundaries:
                      min                 max     size
split                                                 
test  2022-09-09 03:17:00 2022-09-18 16:18:00   761639
train 2022-09-01 00:00:00 2022-09-07 14:55:00  3554957
val   2022-09-07 14:56:00 2022-09-09 03:16:00   761749

01_audit sorted by time?  False
split_idx sorted by time? True
Timestamps identical row-by-row? False
Same multiset of timestamps? True


In [12]:
# D7 test 4 (final): wind-down rule verified by execution, entirely on
# 03_split_index (time-sorted, self-contained). Constants: median over ALL
# 18 days = 207,406; the previously recorded 482,369.5 was from a superseded
# filtered computation. Classification is unchanged under the honest constants.
import pandas as pd
import datetime

split_idx = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\03_split_index.parquet")

daily = split_idx.groupby(split_idx["Timestamp"].dt.date).size()
M = daily.median()
threshold = 0.05 * M

winddown_days = set(daily[daily < threshold].index)
expected = {datetime.date(2022, 9, d) for d in range(11, 19)}
trainval_days = set(split_idx.loc[split_idx["split"].isin(["train", "val"]),
                                  "Timestamp"].dt.date)

assert len(split_idx) == 5_078_345
assert M == 207406.0, f"Median mismatch: {M}"
assert winddown_days == expected, f"Wind-down set changed: {sorted(winddown_days)}"
assert daily[sorted(winddown_days)].sum() == 1108, "Wind-down row count mismatch"
assert not (trainval_days & winddown_days), f"FAIL: {trainval_days & winddown_days}"
print(f"PASS: M={M:,.1f}, threshold={threshold:,.1f}, wind-down = 11-18 Sep 2022, "
      f"1,108 rows, no training or validation day affected.")

PASS: M=207,406.0, threshold=10,370.3, wind-down = 11-18 Sep 2022, 1,108 rows, no training or validation day affected.


In [14]:
# Step 1: list what actually exists in data/processed, with sizes.
from pathlib import Path

proc = Path(r"C:\Fintech-Project\graph_AML_pipeline\data\processed")
for p in sorted(proc.iterdir()):
    size_mb = p.stat().st_size / 1e6 if p.is_file() else 0
    kind = "DIR " if p.is_dir() else "file"
    print(f"{kind}  {size_mb:8.1f} MB  {p.name}")

file     133.1 MB  01_audit.parquet
file       0.2 MB  03_split_index.parquet
file       5.8 MB  04_account_map.parquet
file      77.9 MB  04_graph_train.pkl


In [17]:
# Notebook-04 integrity check via fingerprint: the training graph's edge count,
# vertex count and total edge weight must equal the values computed directly
# from the audit file's training window (time-defined, so no join can mislead).
# Weight comparison uses RELATIVE tolerance: summing 3.5M float64 values in
# two different orders differs at ~1e-13 relative, which is float noise.
import pandas as pd
import pickle
import math

# Ground truth from the audit file, by time alone
df = pd.read_parquet(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\01_audit.parquet",
                     columns=["Timestamp", "Amount Received"])
cutoff = pd.Timestamp("2022-09-07 14:55:00")
train_mask = df["Timestamp"] <= cutoff
n_train = int(train_mask.sum())
w_train = float(df.loc[train_mask, "Amount Received"].sum())
print(f"Audit file, train window: {n_train:,} rows, "
      f"total Amount Received = {w_train:,.2f}")

# The graph as built by notebook 04
with open(r"C:\Fintech-Project\graph_AML_pipeline\data\processed\04_graph_train.pkl", "rb") as f:
    g = pickle.load(f)

print(f"Graph: {g.vcount():,} vertices, {g.ecount():,} edges")
w_graph = float(sum(g.es["weight"]))
print(f"Graph total edge weight = {w_graph:,.2f}")

assert n_train == 3_554_957, f"Train-window row count: {n_train}"
assert g.vcount() == 513_284, f"Vertex count: {g.vcount()}"
assert g.ecount() == n_train, "Edge count != train-window row count"
assert math.isclose(w_graph, w_train, rel_tol=1e-9), \
    f"Weight mismatch beyond float tolerance: {w_graph} vs {w_train}"
print("PASS: graph edge count, vertex count and total edge weight match the "
      "time-defined training window (weights within float64 tolerance). "
      "Notebook 04 consumed the split correctly.")

Audit file, train window: 3,554,957 rows, total Amount Received = 23,436,433,019,029.64
Graph: 513,284 vertices, 3,554,957 edges
Graph total edge weight = 23,436,433,019,027.12
PASS: graph edge count, vertex count and total edge weight match the time-defined training window (weights within float64 tolerance). Notebook 04 consumed the split correctly.
